# Sample Notebook

This notebook loads the active Azure Developer CLI environment. To use a local `.env` file instead, update the first code cell.

In [ ]:
from io import StringIO
from subprocess import run

from dotenv import load_dotenv

result = run(
    ["azd", "env", "get-values"],
    capture_output=True,
    check=False,
    text=True,
)
if result.returncode == 0:
    print("Found AZD environment. Loading...")
    load_dotenv(stream=StringIO(result.stdout), override=True)

load_dotenv(override=True)

## Connect to a Microsoft Foundry project

This example uses the project endpoint and deployment name exported by the Bicep template: `AI_FOUNDRY_PROJECT_ENDPOINT` and `AI_FOUNDRY_DEPLOYMENT_NAME`. Run `azd up` first so those values exist in the active AZD environment.

It follows the current Foundry SDK pattern: create an `AIProjectClient` with `DefaultAzureCredential()`, get an OpenAI-compatible client from the project, and call the Responses API.

In [ ]:
import os

from azure.ai.projects import AIProjectClient
from azure.identity import DefaultAzureCredential

project_endpoint = os.environ["AI_FOUNDRY_PROJECT_ENDPOINT"]
model_deployment_name = os.environ["AI_FOUNDRY_DEPLOYMENT_NAME"]

with (
    DefaultAzureCredential() as credential,
    AIProjectClient(endpoint=project_endpoint, credential=credential) as project_client,
    project_client.get_openai_client() as openai_client,
):
    response = openai_client.responses.create(
        model=model_deployment_name,
        input="Write me a poem about flowers",
    )

print(response.output_text)